<a href="https://colab.research.google.com/github/vinaykumar56/agentic-ai/blob/main/Problem3_RoutedSupportAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Routed Support Agent with MCP Tool Invocation - Guided Notebook

## Problem Statement

Build a Routed Support Agent using:

- LangGraph for workflow orchestration
- MCP (Model Context Protocol) for tool invocation
- Langfuse for tracing and observability

### Workflow

User Query → Classify → Route → Invoke MCP Tool → Respond

---

## Your Tasks

### Build
- Create a multi-step workflow using LangGraph
- Add classification and routing logic
- Invoke tools through MCP
- Generate final responses

### AgenticOps Tasks
- Trace the workflow using Langfuse
- Evaluate:
  - Routing accuracy
  - Tool usage
  - Unnecessary steps
- Improve workflow logic after observing traces

### Final Output Required
- Workflow Diagram
- Trace Logs



# Step 1 - Install Required Packages
Install all libraries required for:
- LangGraph
- MCP
- Azure OpenAI
- Langfuse


In [2]:

# TODO:
# Install required packages

# Hint:
!pip install langfuse
!pip install fastmcp
!pip install langchain_openai
!pip install langchain_mcp_adapters
!pip install langgraph


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.2.1
    Uninstalling wrapt-2.2.1:
      Successfully uninstalled wrapt-2.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.0/219.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.9/272.9 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 

In [13]:
!pip install langchain-huggingface


# Step 2 - Import Libraries

Import all required modules for:
- LangGraph
- MCP Server
- MCP Client
- Langfuse
- Azure OpenAI


In [3]:

# TODO:
# Import required libraries

# Helpful imports:

from langfuse import Langfuse
from langfuse.langchain import CallbackHandler
from langfuse import observe

from fastmcp import FastMCP

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

from langchain_mcp_adapters.client import MultiServerMCPClient

import threading
import os



# Step 3 - Configure Environment Variables

Configure:
- Azure OpenAI credentials
- Langfuse credentials

Also initialize:
- Langfuse client
- Langfuse callback handler


In [7]:
import os
from getpass import getpass
# TODO:
# Configure environment variables

os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass('Enter HUGGING FACE API Key:')

# Initialize Langfuse
# Create CallbackHandler
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("pk_YOUR_LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = getpass("sk_YOUR_LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

Enter HUGGING FACE API Key:··········
pk_YOUR_LANGFUSE_PUBLIC_KEY··········
sk_YOUR_LANGFUSE_SECRET_KEY··········


In [11]:
# langfuse = Langfuse()
langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com"

)

auth_ok = langfuse.auth_check
if auth_ok:
    print("Authentication successful")
else:
    print("Authentication failed")
    exit(1)

# Initialize Langfuse callback handler
langfuse_handler = CallbackHandler()

Authentication successful



# Step 4 - Initialize the LLM

Create an AzureChatOpenAI model.

### Suggested Configuration
- temperature = 0
- Use your deployment name


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

model_id = "google/gemma-3-1b-it"

# 1. Load the tokenizer and model locally using Hugging Face transformers
tokenizer = AutoTokenizer.from_pretrained(model_id, token=os.getenv("HUGGINGFACEHUB_API_TOKEN"))
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

# 2. Create the Hugging Face Pipeline explicitly for text-generation
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7
)

# 3. Wrap it in LangChain's pipeline object
base_llm = HuggingFacePipeline(pipeline=hf_pipe)

# 4. Wrap with ChatHuggingFace so it formats prompts as ChatMessages correctly
llm = ChatHuggingFace(llm=base_llm)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



# Step 5 - Create the MCP Server

Create an MCP server named:

`OrderServer`

This server should expose support-related tools.


In [17]:

# TODO:
# Create MCP server

# Hint:
mcp = FastMCP("OrderServer")



# Step 6 - Register MCP Tools

Example tools:

### Tool
`get_order_status_tool(order_id)`


---

### Tool
`get_return_status_tool(return_id)`



In [18]:
@mcp.tool()
def get_order_status_tool(order_id: str):
    """Returns the status of an order given its ID.

    Args:
        order_id (str): The ID of the order.

    Returns:
        str: The status of the order.
    """
    # Placeholder for actual order status logic
    if order_id == "123":
        return {"order_id": order_id, "status": "shipped", "estimated_delivery": "2024-05-20"}
    elif order_id == "456":
        return {"order_id": order_id, "status": "processing"}
    else:
        return {"order_id": order_id, "status": "not found"}

@mcp.tool()
def get_return_status_tool(return_id: str):
    """Returns the status of a return given its ID.

    Args:
        return_id (str): The ID of the return.

    Returns:
        str: The status of the return.
    """
    # Placeholder for actual return status logic
    if return_id == "R789":
        return {"return_id": return_id, "status": "received", "refund_processed": True}
    elif return_id == "R101":
        return {"return_id": return_id, "status": "initiated"}
    else:
        return {"return_id": return_id, "status": "not found"}


# Step 7 - Start MCP Server

Run the MCP server in a background thread.


In [19]:
def run_server():
    mcp.run()

# Start server in background thread
server_thread = threading.Thread(target=run_server)
server_thread.daemon = True # Allow main program to exit even if server thread is still running
server_thread.start()


# Step 8 - Create MCP Client

Create an MCP client that connects to:

`http://127.0.0.1:8000/mcp`

Then:
- Load tools
- Create ToolNode
- Bind tools with LLM


In [41]:
from langgraph.prebuilt import ToolNode
import asyncio

# 1. Create client
mcp_client = MultiServerMCPClient( {"DemoServer":{"url":"http://127.0.0.1:8000/mcp",
                                   "transport":"http"}})

# 2. Load tools
# mcp_tools = mcp_client.get_langchain_tools()
# mcp_tools = mcp_client.get_tools()
# dir(mcp_client)


# 2. Load tools (await the coroutine)
mcp_tools = await mcp_client.get_tools()  # or await mcp_client.load_tools()

# 3. Create ToolNode
mcp_tool_node = ToolNode(mcp_tools)

# 4. Bind tools to LLM
llm_with_tools = llm.bind_tools(mcp_tools, stop=["<|eot_id|>"])

ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)


# Step 9 - Create Order Processing Node

This node should:
- Receive graph state
- Invoke the LLM with tools
- Return messages


In [42]:
def order_node(state):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# Step 10 - Create General Response Node

This node handles:
- Non-order-related queries

Requirements:
- Respond briefly
- If order/package related queries reach this node,
  tell the user you cannot help


In [43]:
def general_response_node(state):
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# Step 11 - Create Initial Classifier Prompt (V1)

Create a basic classifier prompt - this could mis-classify the query.


In [44]:
System_prompt1 = (
    "You are a helpful assistant that classifies user queries.\n"
    "Classify the following query as 'order_query' if it relates to order status or return status.\n"
    "Otherwise, classify it as 'general_query'.\n"
    "Return only one word in lowercase, for example: 'order_query' or 'general_query'."
)


# Step 12 - Create Classification Node

Create:
- classify_query(state)
- route_decision(state)

The classifier should:
- Read latest user message
- Invoke LLM
- Return routing decision


In [45]:
def classify_query(state):
    messages = state["messages"]
    latest_message = messages[-1]

    # Prepare the prompt for the classifier LLM
    classification_prompt = [
        HumanMessage(content=System_prompt1),
        HumanMessage(content=latest_message.content)
    ]

    # Invoke the LLM to classify the query
    response = llm.invoke(classification_prompt)
    classification = response.content.strip().lower()

    return {"classification": classification}

def route_decision(state):
    classification = state["classification"]
    if classification == "order_query":
        return "order_node"
    else:
        return "general_response_node"


# Step 13 - Build LangGraph Workflow

Construct the workflow:

START
  ↓
Classifier
  ↓
Conditional Routing
  ├── Order Node
  │      ↓
  │   Tool Node
  │      ↓
  │   Order Node
  │
  └── General Node
          ↓
         END


In [47]:
# 1. Create graph
graph = StateGraph(MessagesState)

# 2. Add nodes
graph.add_node("classify", classify_query)
graph.add_node("order_node", order_node)
# graph.add_node("tool_node", mcp_tool_node) # Using the ToolNode created in Step 8
graph.add_node("general_response_node", general_response_node)

# 3. Set entry point
graph.set_entry_point("classify")

# 4. Add conditional edges
graph.add_conditional_edges(
    "classify",
    route_decision, # Routes based on 'order_query' or 'general_query'
    {
        "order_node": "order_node",
        "general_response_node": "general_response_node",
    },
)

# If the order node invokes a tool, go to the tool_node. Otherwise, end.
graph.add_conditional_edges(
    "order_node",
    tools_condition, # Checks if tools were invoked by the LLM
    {
        "tools": "tool_node", # If tools were invoked, go to the tool_node
        "__end__": END,      # If no tools, it's a final answer, so end
    },
)

# After tools are executed, return to the order_node to process the tool output
graph.add_edge("tool_node", "order_node")

# General response node always leads to the end
graph.add_edge("general_response_node", END)

# 5. Compile graph
app = graph.compile(checkpointer=None, interrupt_after=[mcp_tool_node.name])

NameError: name 'mcp_tool_node' is not defined


# Step 14 - Visualize Workflow

Display the workflow graph.

### Expected Output
A workflow diagram showing:
- classifier
- order node
- tool node
- general node


In [48]:
# TODO:
# Display graph image

# Hint:
app.get_graph().draw_mermaid_png()

NameError: name 'app' is not defined


# Step 15 - Test Query 1

Test with a correctly classified example:


Observe:
- Classification
- Tool invocation
- Final response


In [49]:
inputs = {"messages": [HumanMessage(content="What is the status of order 123?")]}
response = app.invoke(inputs, config={"callbacks": [langfuse_handler]})
display(response)

NameError: name 'app' is not defined


# Step 16 - Test Query 2

Test with a wrongly classified example:

Observe the behavior carefully.

### Think
Did the classifier route correctly?


In [50]:

# TODO:
# Invoke workflow

# Check Langfuse


# Step 17 - Analyze Langfuse Traces

Open Langfuse and inspect:

- Trace flow
- Node execution
- Routing decision
- Tool usage

## Questions
1. Was the query routed correctly?
2. Did the classifier fail?



# Step 18 - Improve Classifier Prompt (V2)

Improve the classifier logic.


In [ ]:

# TODO:
# Create improved System_prompt2



# Step 19 - Create Improved Classifier Node

Create:
- classify_query_v2(state)


In [ ]:

# TODO:
# Create classify_query_v2(state)



# Step 20 - Build Improved Workflow

Rebuild the graph using:
- classify_query_v2


In [ ]:

# TODO:
# Build improved graph
# Compile as app2



# Step 21 - Re-Test Workflow

Test again


Verify:
- Correct routing
- Tool invocation


In [ ]:

# TODO:
# Invoke app2


### Check LangFuse again to see the classification

